In [1]:
import requests
from requests.auth import HTTPBasicAuth

# El '0' indica "mi usuario actual"
url = "https://intervals.icu/api/v1/athlete/0/athlete-summary"

# Reemplaza con tu clave real obtenida en /settings
api_key = "1347lyp9p4meza7lyd3sar4qu" 

# La autenticación usa 'API_KEY' como usuario y tu clave como contraseña
auth = HTTPBasicAuth('API_KEY', api_key)

response = requests.get(url, auth=auth)

if response.status_code == 200:
    atletas = response.json()
    # Quitar duplicados por athlete_id y nombre
    atletas = list({(a.get('athlete_id'), a.get('athlete_name')): a for a in atletas}.values())
    print(f"Se encontraron {len(atletas)} atletas únicos.")
    print(atletas)
else:
    print(f"Error: {response.status_code}")

Se encontraron 9 atletas únicos.
[{'count': 4, 'time': 23440, 'moving_time': 23440, 'elapsed_time': 24135, 'calories': 5609, 'total_elevation_gain': 3947.0, 'training_load': 611, 'srpe': 0, 'distance': 215290.98, 'eftp': None, 'eftpPerKg': None, 'date': '2026-04-06', 'athlete_id': 'i547906', 'athlete_name': 'Adrian Fajardo', 'email': None, 'external_id': None, 'fitness': 11.740628, 'fatigue': 66.427925, 'form': -54.687298, 'rampRate': None, 'weight': 64.0, 'timeInZones': [6554, 1856, 3511, 3335, 3479, 3919, 788, 3723], 'timeInZonesTot': 23442, 'byCategory': [{'count': 4, 'time': 23440, 'moving_time': 23440, 'elapsed_time': 24135, 'calories': 5609, 'total_elevation_gain': 3947.0, 'training_load': 611, 'srpe': 0, 'distance': 215290.98, 'eftp': 290.0, 'eftpPerKg': 4.53125, 'category': 'Ride'}], 'mostRecentWellnessId': '2026-04-07'}, {'count': 5, 'time': 48252, 'moving_time': 48252, 'elapsed_time': 54279, 'calories': 11796, 'total_elevation_gain': 6721.0, 'training_load': 650, 'srpe': 0, '

In [2]:
print("\n=== NOMBRES DE ATLETAS ===")
for atleta in atletas:
    print(f"- {atleta['athlete_name']} - id {atleta['athlete_id']}")


=== NOMBRES DE ATLETAS ===
- Adrian Fajardo - id i547906
- CarlosGarcia11 - id i495562
- Iosman - id i419268
- Jose Luis Faura - id i547904
- Josemanueldiaz - id i547157
- LorenzoQ - id i545517
- Sinuhé - id i545516
- david Echavarri - id 208681
- oka_ander - id i546414


In [3]:
from datetime import datetime, timedelta
print("\n=== ESTADÍSTICAS DE ENTRENAMIENTO ===")
for i, atleta in enumerate(atletas, 1):
    nombre = atleta['athlete_name']
    fecha = atleta['date']
    distancia_m = atleta['distance']  # en metros
    distancia_km = distancia_m / 1000  # convertir a km
    tiempo_s = atleta['moving_time']  # en segundos
    calorias = atleta.get('calories', 0)
    kilojulios = calorias * 4.184  # 1 kcal = 4.184 kJ
    peso = atleta.get('weight')
    print(f"\n{i}. {nombre} - Fecha: {fecha}")
    # Convertir tiempo a hh:mm:ss
    horas = tiempo_s // 3600
    minutos = (tiempo_s % 3600) // 60
    segundos = tiempo_s % 60
    tiempo_formateado = f"{int(horas):02d}:{int(minutos):02d}:{int(segundos):02d}"
    
    # Calcular velocidad media en km/h
    tiempo_h = tiempo_s / 3600
    velocidad_media = distancia_km / tiempo_h if tiempo_h > 0 else 0
    
    # Calcular trabajo/peso (en kJ/kg)
    trabajo_por_peso = kilojulios / peso if peso and peso > 0 else 65

    print(f"{nombre}, {fecha}, {distancia_km:.2f}km, {trabajo_por_peso:.2f}kj/kg")
    
    # Configurar rango de fechas (últimos 90 días)
    oldest = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
    newest = datetime.now().strftime('%Y-%m-%d')

    url_actividades = f"https://intervals.icu/api/v1/athlete/{atleta['athlete_id']}/activities?oldest={oldest}&newest={newest}"

    response_actividades = requests.get(url_actividades, auth=auth)

    if response_actividades.status_code == 200:
        actividades = response_actividades.json()
        print(f"\n=== ACTIVIDADES DEL ATLETA ===")
        print(f"Se encontraron {len(actividades)} actividades.\n")
    
        for i, actividad in enumerate(actividades, 1):
            
            fecha = actividad.get('start_date_local', 'N/A')
            nombre = actividad.get('name', 'Sin nombre')
            tipo_deporte = actividad.get('type', 'N/A')
            distancia = (actividad.get('distance') or 0) / 1000  # convertir a km
            tiempo = actividad.get('moving_time') or 0
        
            # Convertir tiempo a hh:mm:ss
            horas_a = tiempo // 3600
            minutos_a = (tiempo % 3600) // 60
            segundos_a = tiempo % 60
            tiempo_formateado_a = f"{int(horas_a):02d}:{int(minutos_a):02d}:{int(segundos_a):02d}"
        
            # Calcular velocidad o ritmo según el tipo de deporte
            if tipo_deporte == 'Run' and distancia > 0:
                # Para Run: calcular ritmo en min/km
                minutos_por_km = tiempo / 60 / distancia
                min_ritmo = int(minutos_por_km)
                seg_ritmo = int((minutos_por_km - min_ritmo) * 60)
                ritmo_str = f"{min_ritmo}:{seg_ritmo:02d} min/km"
                velocidad_str = ritmo_str
            else:
                # Para otros deportes: velocidad en km/h
                tiempo_h_a = tiempo / 3600
                velocidad = (distancia / tiempo_h_a) if tiempo_h_a > 0 else 0
                velocidad_str = f"{velocidad:.2f} km/h"
        
            print(f"{i}. {fecha} - {nombre} ({tipo_deporte})")
            print(f"   Distancia: {distancia:.2f} km | Tiempo: {tiempo_formateado_a} | Ritmo/Velocidad: {velocidad_str}")
        
            # Obtener detalles completos de la actividad para interval_summary
            id_actividad = actividad.get('id')
            url_detalle = f"https://intervals.icu/api/v1/athlete/{atleta['athlete_id']}/activities/{id_actividad}"
            response_detalle = requests.get(url_detalle, auth=auth)
            
            if response_detalle.status_code == 200:
                detalle_act = response_detalle.json()
                if isinstance(detalle_act, list) and len(detalle_act) > 0:
                    detalle_act = detalle_act[0]
                
                # Mostrar interval_summary si existe
                if 'interval_summary' in detalle_act and detalle_act['interval_summary']:
                    print(f"   Intervalos: {', '.join(detalle_act['interval_summary'])}")
            
                print()
        
            else:
                print(f"Error al obtener actividades: {response_actividades.status_code}")
                print(response_actividades.text)


=== ESTADÍSTICAS DE ENTRENAMIENTO ===

1. Adrian Fajardo - Fecha: 2026-04-06
Adrian Fajardo, 2026-04-06, 215.29km, 366.69kj/kg

=== ACTIVIDADES DEL ATLETA ===
Se encontraron 4 actividades.

1. 2026-04-07T13:09:36 - Pamplona Ciclismo en ruta (Ride)
   Distancia: 172.99 km | Tiempo: 04:52:34 | Ritmo/Velocidad: 35.48 km/h
   Intervalos: 39x 8s 440w, 4x 28s 359w, 8x 2m6s 263w, 40x 42s 297w, 1x 1m46s 303w, 1x 3m24s 314w, 1x 3m3s 308w, 1x 86s 310w, 1x 70s 330w, 1x 2m36s 312w, 1x 5m14s 326w, 1x 22s 389w, 1x 19s 402w, 1x 6m16s 329w, 1x 6m56s 261w, 1x 38s 389w, 1x 3m18s 275w, 1x 92s 233w, 3x 2m51s 261w, 1x 4m13s 236w, 1x 77s 277w

2. 2026-04-06T14:43:58 - Bilbao Ciclismo en ruta (Ride)
   Distancia: 13.61 km | Tiempo: 00:19:43 | Ritmo/Velocidad: 41.43 km/h
   Intervalos: 1x 19m47s 181bpm

3. 2026-04-06T14:08:13 - Bilbao Ciclismo en ruta (Ride)
   Distancia: 0.00 km | Tiempo: 00:23:09 | Ritmo/Velocidad: 0.00 km/h
   Intervalos: 1x 24m45s 130bpm

4. 2026-04-06T11:59:58 - Bilbao Ciclismo en ruta 

In [4]:
# Obtener las actividades del atleta


# Configurar rango de fechas (últimos 90 días)
oldest = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')
newest = datetime.now().strftime('%Y-%m-%d')

url_actividades = f"https://intervals.icu/api/v1/athlete/0/activities?oldest={oldest}&newest={newest}"

response_actividades = requests.get(url_actividades, auth=auth)

if response_actividades.status_code == 200:
    actividades = response_actividades.json()
    print(f"\n=== ACTIVIDADES DEL ATLETA ===")
    print(f"Se encontraron {len(actividades)} actividades.\n")
    
    for i, actividad in enumerate(actividades, 1):
        print("--------------------------------------------------"+str(actividad))
        fecha = actividad.get('start_date_local', 'N/A')
        nombre = actividad.get('name', 'Sin nombre')
        tipo_deporte = actividad.get('type', 'N/A')
        distancia = (actividad.get('distance') or 0) / 1000  # convertir a km
        tiempo = actividad.get('moving_time') or 0
        
        # Convertir tiempo a hh:mm:ss
        horas_a = tiempo // 3600
        minutos_a = (tiempo % 3600) // 60
        segundos_a = tiempo % 60
        tiempo_formateado_a = f"{int(horas_a):02d}:{int(minutos_a):02d}:{int(segundos_a):02d}"
        
        # Calcular velocidad o ritmo según el tipo de deporte
        if tipo_deporte == 'Run' and distancia > 0:
            # Para Run: calcular ritmo en min/km
            minutos_por_km = tiempo / 60 / distancia
            min_ritmo = int(minutos_por_km)
            seg_ritmo = int((minutos_por_km - min_ritmo) * 60)
            ritmo_str = f"{min_ritmo}:{seg_ritmo:02d} min/km"
            velocidad_str = ritmo_str
        else:
            # Para otros deportes: velocidad en km/h
            tiempo_h_a = tiempo / 3600
            velocidad = (distancia / tiempo_h_a) if tiempo_h_a > 0 else 0
            velocidad_str = f"{velocidad:.2f} km/h"
        
        print(f"{i}. {fecha} - {nombre} ({tipo_deporte})")
        print(f"   Distancia: {distancia:.2f} km | Tiempo: {tiempo_formateado_a} | Ritmo/Velocidad: {velocidad_str}")
        
        # Obtener detalles completos de la actividad para interval_summary
        id_actividad = actividad.get('id')
        url_detalle = f"https://intervals.icu/api/v1/athlete/{id}/activities/{id_actividad}"
        response_detalle = requests.get(url_detalle, auth=auth)
        
        if response_detalle.status_code == 200:
            detalle_act = response_detalle.json()
            if isinstance(detalle_act, list) and len(detalle_act) > 0:
                detalle_act = detalle_act[0]
            
            # Mostrar interval_summary si existe
            if 'interval_summary' in detalle_act and detalle_act['interval_summary']:
                print(f"   Intervalos: {', '.join(detalle_act['interval_summary'])}")
        
        print()
        
else:
    print(f"Error al obtener actividades: {response_actividades.status_code}")
    print(response_actividades.text)


=== ACTIVIDADES DEL ATLETA ===
Se encontraron 8 actividades.

--------------------------------------------------{'id': '18014998431', 'icu_athlete_id': 'i547157', 'start_date_local': '2026-04-07T18:02:19', 'source': 'STRAVA', '_note': 'STRAVA activities are not available via the API'}
1. 2026-04-07T18:02:19 - Sin nombre (N/A)
   Distancia: 0.00 km | Tiempo: 00:00:00 | Ritmo/Velocidad: 0.00 km/h

--------------------------------------------------{'id': '18015010024', 'icu_athlete_id': 'i547157', 'start_date_local': '2026-04-07T13:09:56', 'source': 'STRAVA', '_note': 'STRAVA activities are not available via the API'}
2. 2026-04-07T13:09:56 - Sin nombre (N/A)
   Distancia: 0.00 km | Tiempo: 00:00:00 | Ritmo/Velocidad: 0.00 km/h

--------------------------------------------------{'id': '18001258576', 'icu_athlete_id': 'i547157', 'start_date_local': '2026-04-06T16:11:58', 'source': 'STRAVA', '_note': 'STRAVA activities are not available via the API'}
3. 2026-04-06T16:11:58 - Sin nombre (N/

In [5]:
# Obtener mejores potencias (5', 10', 20') de actividades
print("\n=== MEJORES ESFUERZOS DE POTENCIA ===\n")

# Primero, contar cuántas actividades tienen datos de potencia
actividades_con_potencia = [a for a in actividades if a.get('avg_watts') and a.get('avg_watts') > 0]
print(f"Actividades con datos de potencia: {len(actividades_con_potencia)} de {len(actividades)}\n")

if not actividades_con_potencia:
    print("No se encontraron actividades con datos de potencia.")
else:
    # Intervalos de tiempo en segundos
    intervalos = [300, 600, 1200]  # 5min, 10min, 20min
    
    for i, actividad in enumerate(actividades_con_potencia[:5], 1):  # Mostrar las primeras 5
        id_actividad = actividad.get('id')
        fecha = actividad.get('start_date_local', 'N/A')
        nombre = actividad.get('name', 'Sin nombre')
        tipo_deporte = actividad.get('type', 'N/A')
        avg_watts = actividad.get('avg_watts')
        
        # Obtener detalles completos de la actividad
        url_detalle = f"https://intervals.icu/api/v1/athlete/0/activities/{id_actividad}"
        response_detalle = requests.get(url_detalle, auth=auth)
        print(f"Actividad ID: {id_actividad}")
        print(response_detalle.text)
        if response_detalle.status_code == 200:
            detalle = response_detalle.json()
            
            print(f"{i}. {fecha} - {nombre} ({tipo_deporte})")
            print(f"   Potencia media: {avg_watts:.0f}W")
            
            # Buscar los mejores esfuerzos en icu_intervals
            if 'icu_intervals' in detalle and detalle['icu_intervals']:
                potencias = {}
                for interval in detalle['icu_intervals']:
                    secs = interval.get('secs')
                    watts = interval.get('watts')
                    if secs in intervalos and watts:
                        tiempo_min = secs // 60
                        potencias[f"{tiempo_min}min"] = watts
                
                if potencias:
                    potencias_str = ' | '.join([f"{k}: {v:.0f}W" for k, v in sorted(potencias.items())])
                    print(f"   Mejores esfuerzos: {potencias_str}")
                else:
                    print("   Mejores esfuerzos: No disponibles en icu_intervals")
            else:
                print("   Mejores esfuerzos: No disponibles")
            
            print()
        else:
            print(f"   Error al obtener detalles: {response_detalle.status_code}\n")


=== MEJORES ESFUERZOS DE POTENCIA ===

Actividades con datos de potencia: 0 de 8

No se encontraron actividades con datos de potencia.


In [6]:
# Buscar mejores 10 minutos de potencia en response_detalle
if response_detalle.status_code == 200:
    detalle = response_detalle.json()
    
    print("=== BÚSQUEDA DE MEJORES 10 MINUTOS DE POTENCIA ===\n")
    print(f"Tipo de respuesta: {type(detalle)}")
    
    # Si es una lista, trabajar con el primer elemento
    if isinstance(detalle, list):
        print(f"La respuesta es una lista con {len(detalle)} elementos")
        if len(detalle) > 0:
            detalle = detalle[0]
            print(f"Trabajando con el primer elemento\n")
        else:
            print("La lista está vacía")
            detalle = None
    
    if detalle:
        # Buscar en icu_intervals
        if 'icu_intervals' in detalle and detalle['icu_intervals']:
            print(f"Total de intervalos en icu_intervals: {len(detalle['icu_intervals'])}\n")
            
            # Buscar específicamente 600 segundos (10 minutos)
            for interval in detalle['icu_intervals']:
                secs = interval.get('secs')
                if secs == 600:  # 10 minutos
                    watts = interval.get('watts')
                    print(f"✓ Encontrado: 10 minutos (600 seg)")
                    print(f"  Potencia: {watts}W")
                    print(f"  Datos completos del intervalo: {interval}")
                    break
            else:
                print("✗ No se encontró intervalo de 10 minutos (600 seg)")
                print("\nIntervalos disponibles (primeros 20):")
                for interval in detalle['icu_intervals'][:20]:
                    secs = interval.get('secs')
                    watts = interval.get('watts')
                    print(f"  {secs}seg ({secs//60}min {secs%60}seg): {watts}W")
        else:
            print("No hay campo 'icu_intervals' en la respuesta")
            print("\nCampos disponibles en la respuesta:")
            if isinstance(detalle, dict):
                for campo in sorted(detalle.keys()):
                    if 'power' in campo.lower() or 'watts' in campo.lower() or 'interval' in campo.lower():
                        print(f"  - {campo}: {detalle[campo]}")
            else:
                print(f"Estructura inesperada: {type(detalle)}")
                print(detalle)
else:
    print(f"Error en response_detalle: {response_detalle.status_code}")
    print(response_detalle.text)

Error en response_detalle: 403
{"status":403,"error":"Access denied"}


In [7]:
# Analizar intervalos de potencia de la actividad
if response_detalle.status_code == 200:
    detalle = response_detalle.json()
    if isinstance(detalle, list) and len(detalle) > 0:
        detalle = detalle[0]
    
    print("=== ANÁLISIS DE INTERVALOS DE POTENCIA ===\n")
    
    # Información general de potencia
    if 'icu_average_watts' in detalle:
        print(f"Potencia media: {detalle['icu_average_watts']}W")
    if 'icu_weighted_avg_watts' in detalle:
        print(f"Potencia normalizada: {detalle['icu_weighted_avg_watts']}W")
    
    # Analizar interval_summary
    if 'interval_summary' in detalle and detalle['interval_summary']:
        print(f"\nResumen de intervalos encontrados:")
        for intervalo in detalle['interval_summary']:
            print(f"  • {intervalo}")
            
            # Buscar el intervalo cercano a 10 minutos
            if '10m' in intervalo:
                print(f"    ⭐ Este es cercano a 10 minutos!")
    
    print("\n" + "="*50)
    print("MEJOR ESFUERZO DE ~10 MINUTOS:")
    # Extraer el intervalo de ~10 minutos
    for intervalo in detalle.get('interval_summary', []):
        if '10m' in intervalo:
            print(f"  {intervalo}")
            # Intentar extraer la potencia
            import re
            match = re.search(r'(\d+)w', intervalo)
            if match:
                potencia_10min = match.group(1)
                print(f"  Potencia: {potencia_10min}W")
    print("="*50)
else:
    print(f"Error: {response_detalle.status_code}")

Error: 403
